# ⚖️ Re-Embedding Dataset Hukumonline dengan Multilingual E5-Large
**Mata Kuliah:** Natural Language Processing (UAS)

Notebook ini melakukan proses embedding terhadap 7.172 dokumen konsultasi hukum dari Klinik Hukumonline.
* **Model:** `intfloat/multilingual-e5-large` (Microsoft XLM-RoBERTa Large, 1024 dimensi)
* **Field Representasi:** `title + question + summarize`
* **Arsitektur E5:** Wajib menyertakan prefix `passage: ` untuk korpus dokumen
* **Output:** `vectors.npy` (matriks vektor) dan `documents.json` (metadata bersih)

## 1. Install & Import Dependensi

In [ ]:
!pip install -q sentence-transformers torch tqdm numpy

In [ ]:
import os
import json
import time
import zipfile
import urllib.request
import torch
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

# Cek ketersediaan GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device aktif: {device.upper()}")
if device == "cuda":
    print(f"GPU Model   : {torch.cuda.get_device_name(0)}")
else:
    print("Perhatian: Berjalan di CPU. Jika menggunakan Colab, aktifkan GPU T4 di Runtime -> Change runtime type -> T4 GPU agar proses selesai dalam ~2-3 menit!")

## 2. Memuat Dataset
Jika file belum ada di lingkungan Colab, notebook akan otomatis mengunduh `embedings.zip` dari repositori GitHub Anda.

In [ ]:
# Otomatis unduh jika belum ada di container Colab
if not os.path.exists("embedings.json") and not os.path.exists("embedings.zip"):
    print("Dataset belum ada di container Colab. Mengunduh embedings.zip otomatis dari GitHub...")
    url = "https://github.com/sammdavid/UAS-Natural-Language-Processing/raw/main/embedings.zip"
    urllib.request.urlretrieve(url, "embedings.zip")
    print("Download selesai!")

raw_data = None
if os.path.exists("embedings.json"):
    print("Membaca langsung dari embedings.json...")
    with open("embedings.json", "r", encoding="utf-8") as f:
        raw_data = json.load(f)
elif os.path.exists("embedings.zip"):
    print("Membaca dari arsip embedings.zip...")
    with zipfile.ZipFile("embedings.zip", "r") as z:
        with z.open("embedings.json") as f:
            raw_data = json.load(f)
else:
    raise FileNotFoundError("File embedings.json atau embedings.zip tidak ditemukan!")

print(f"Total data berhasil dimuat: {len(raw_data)} entri")

## 3. Ekstraksi Teks Representasi & Metadata
Sesuai rancangan arsitektur, teks yang di-embed adalah gabungan **`title + question + summarize`** dengan prefix `passage: `.
Field ulasan lengkap (`answer`) disimpan di metadata dokumen untuk digunakan sebagai konteks LLM.

In [ ]:
corpus_passages = []
documents_metadata = []

for idx, item in enumerate(raw_data):
    title = item.get("title", "").strip()
    question = item.get("question", "").strip()
    summarize = item.get("summarize", "").strip()

    # Format wajib E5 untuk dokumen
    passage_text = f"passage: {title}. {question} {summarize}".strip()
    corpus_passages.append(passage_text)

    # Metadata bersih
    documents_metadata.append({
        "id": idx,
        "title": title,
        "question": question,
        "summarize": summarize,
        "answer": item.get("answer", ""),
        "source": item.get("source", "")
    })

print(f"Total passages siap di-encode: {len(corpus_passages)}")
print("\nContoh passage pertama:")
print(corpus_passages[0][:300], "...")

## 4. Memuat Model Multilingual E5-Large
Model resmi Microsoft (`intfloat/multilingual-e5-large`) dengan dimensi vektor **1024**.

In [ ]:
model_name = "intfloat/multilingual-e5-large"
print(f"Mengunduh dan memuat model: {model_name}...")
model = SentenceTransformer(model_name, device=device)
print("Model berhasil dimuat!")

## 5. Komputasi Vektor Embedding
Menghasilkan representasi vektor float32 (L2-normalized) untuk seluruh 7.172 dokumen.

In [ ]:
batch_size = 32 if device == "cuda" else 16
print(f"Memulai embedding dengan batch_size={batch_size}...")
start_time = time.time()

vectors = model.encode(
    corpus_passages,
    batch_size=batch_size,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

elapsed = time.time() - start_time
print(f"Selesai dalam {elapsed:.2f} detik! ({elapsed/60:.2f} menit)")
print(f"Bentuk matriks vektor: {vectors.shape} (Jumlah dokumen x Dimensi)")

## 6. Simpan Hasil (`vectors.npy` & `documents.json`)

In [ ]:
# 1. Simpan matriks vektor (format numpy float32, ukuran ~29 MB)
np.save("vectors.npy", vectors.astype(np.float32))

# 2. Simpan metadata dokumen (format json, ukuran ~15-20 MB)
with open("documents.json", "w", encoding="utf-8") as f:
    json.dump(documents_metadata, f, ensure_ascii=False, indent=2)

print("File berhasil disimpan:")
print(f"- vectors.npy    : {os.path.getsize('vectors.npy') / (1024*1024):.2f} MB")
print(f"- documents.json : {os.path.getsize('documents.json') / (1024*1024):.2f} MB")

## 7. Sanity Check / Uji Coba Pencarian Semantik Sederhana
Menguji vektor baru dengan pertanyaan contoh untuk memverifikasi akurasi retrieval.

In [ ]:
def test_search(query_text, top_k=3):
    # Format wajib E5 untuk pertanyaan
    formatted_q = f"query: {query_text.strip()}"
    q_vec = model.encode(formatted_q, normalize_embeddings=True, convert_to_numpy=True)

    # Cosine similarity via dot product
    scores = np.dot(vectors, q_vec)
    top_ids = np.argsort(scores)[::-1][:top_k]

    print(f"=== Hasil Pencarian: '{query_text}' ===\n")
    for rank, idx in enumerate(top_ids, 1):
        doc = documents_metadata[idx]
        print(f"[{rank}] Skor: {scores[idx]:.4f}")
        print(f"     Judul: {doc['title']}")
        print(f"     Kasus: {doc['question'][:120]}...")
        print(f"     URL  : {doc['source']}\n")

# Uji coba dengan studi kasus hukum nyata
test_search("bagaimana cara membeli tanah girik tanpa sertifikat yang aman?")
test_search("provisions for foreign chef working in indonesia")

## 8. Download Hasil (Jika Menggunakan Google Colab)
Jalankan cell ini jika Anda menjalankan notebook di Google Colab dan ingin mengunduh kedua file hasil ke komputer lokal Anda.

In [ ]:
try:
    from google.colab import files
    print("Mengunduh vectors.npy...")
    files.download("vectors.npy")
    print("Mengunduh documents.json...")
    files.download("documents.json")
except ImportError:
    print("Bukan di Google Colab. File sudah tersimpan di direktori lokal Anda.")